In [ ]:
from hw2fintools import gurufocus as gf
import os
from dotenv import load_dotenv
import pandas as pd
import re
import numpy as np


load_dotenv()
path_stockdata = os.path.join(os.environ.get('judgement_day'), 'Data--StockWatchList')
date_pattern = r"(\d\d\d\d-\d\d-\d\d)--"

ticker_list = os.listdir(path_stockdata)

In [ ]:
for item in ticker_list:
    loc = 0
    if item.startswith('._'):
        ticker_list.remove(item)

ticker_list.sort()

process_df = pd.DataFrame(ticker_list, columns=['Ticker'])
process_df

In [ ]:
process_df['DivFile'] = ''
process_df['DivData'] = False
process_df['PriceFile'] = ''
process_df['PriceData'] = False
process_df['DateMatch'] = ''

process_df

In [ ]:
for index, row in process_df.iterrows():
    path_ticker = os.path.join(path_stockdata, row['Ticker'])
    div_list = []
    price_list = []

    for item in os.listdir(path_ticker):
        if 'gf-raw-dividend_history' in item and item.endswith('.csv') and not item.startswith('._'):
            div_list.append(item)
        elif 'gf-raw-price_history' in item and item.endswith('.csv') and not item.startswith('._'):
            price_list.append(item)

    div_list_sorted = sorted(div_list, reverse=True)
    div_current = div_list_sorted[0]
    process_df.loc[index, 'DivFile'] = div_current

    price_list_sorted = sorted(price_list, reverse=True)
    price_current = price_list_sorted[0]
    process_df.loc[index, 'PriceFile'] = price_current

    div_df = pd.read_csv(os.path.join(path_ticker, div_current))

    div_len = len(div_df)

    if div_len <= 20:
        process_df.loc[index, 'DivData'] = False
    else:
        process_df.loc[index, 'DivData'] = True

    price_df = pd.read_csv(os.path.join(path_ticker, price_current))

    if price_df.empty:
        process_df.loc[index, 'PriceData'] = False
    else:
        process_df.loc[index, 'PriceData'] = True

    div_date = re.search(date_pattern, div_current)
    price_date = re.search(date_pattern, price_current)

    div_match = div_date.group(1)
    price_match = price_date.group(1)

    if div_match == price_match:
        process_df.loc[index, 'DateMatch'] = True
    else:
        process_df.loc[index, 'DateMatch'] = False



In [ ]:
process_df

In [ ]:
mask = (process_df['DivData'] == True) & (process_df['PriceData'] == True) & (process_df['DateMatch'] == True)
auto_run_df = process_df[mask].copy()
auto_run_df

In [ ]:
for index, row in auto_run_df.iterrows():
    path_ticker = os.path.join(path_stockdata, row['Ticker'])
    div_path = os.path.join(path_ticker, row['DivFile'])
    price_path = os.path.join(path_ticker, row['PriceFile'])

    print(row['Ticker'] + '--Start')

    div_df0 = gf.div_hist_s1v1(div_path)
    price_df0 = gf.price_hist_s1v1(price_path)

    div_df1 = div_df0.loc[div_df0['DivType'] == 'regular']
    div_df1 = div_df1.drop(columns=['DivRecordDate', 'DivDeclareDate', 'DivPayDate'])
    div_df1 = div_df1.rename(columns={'ExDivDate': 'Date'})

    merged_df0 = pd.merge(price_df0, div_df1, on='Date', how='left')
    merged_df0['DivAmount'] = merged_df0['DivAmount'].fillna(0)
    merged_df0['DivFrequency'] = merged_df0['DivFrequency'].fillna(div_df1.iloc[0]['DivFrequency'])
    merged_df0['DivType'] = merged_df0['DivType'].fillna(div_df1.iloc[0]['DivType'])
    merged_df0['DivPayDeclared'] = merged_df0['DivAmount'].fillna(0)

    div_var = 0

    for index1, row1 in merged_df0.iterrows():
        if row1['DivAmount'] > 0:
            div_var = row1['DivAmount']

        else:
            merged_df0.at[index1, 'DivAmount'] = div_var

    merged_df0['FwdDiv'] = merged_df0['DivFrequency'] * merged_df0['DivAmount']
    merged_df0['FwdDivYield'] = merged_df0['FwdDiv'] / merged_df0['PricePerShare']

    aggr_df0 = merged_df0
    aggr_df0 = aggr_df0.set_index('Date')
    aggr_df1 = aggr_df0.groupby(aggr_df0.index.year).agg(
        SharePriceMin=pd.NamedAgg(column='PricePerShare', aggfunc='min'),
        SharePriceMax=pd.NamedAgg(column='PricePerShare', aggfunc='max'),
        SharePriceMean=pd.NamedAgg(column='PricePerShare', aggfunc='mean'),
        SharePriceMedian=pd.NamedAgg(column='PricePerShare', aggfunc='median'),
        DivYieldMin=pd.NamedAgg(column='FwdDivYield', aggfunc='min'),
        DivYieldMax=pd.NamedAgg(column='FwdDivYield', aggfunc='max'),
        DivYieldMean=pd.NamedAgg(column='FwdDivYield', aggfunc='mean'),
        DivYieldMedian=pd.NamedAgg(column='FwdDivYield', aggfunc='median'),
        DivPaidTotal=pd.NamedAgg('DivPayDeclared', aggfunc='sum')
    )

    aggr_df1.index.name = 'DateCy'

    merged_df0.to_csv(os.path.join(path_ticker, row['Ticker'].upper() + '--DivPrice_History-s1v1.csv'))
    aggr_df1.to_csv(os.path.join(path_ticker, row['Ticker'].upper() + '--Aggregate_Cy_DivPrice_History-s1v1.csv'))

    print(row['Ticker'] + '--Completed')

